# OPDC Data Preparation

This notebook builds MDS-UPDRS Parts I, II, and III tables for the OPDC
(Oxford Parkinson's Disease Centre) Discovery cohort, reshaped and
renamed to match the PPMI MDS-UPDRS column conventions so both cohorts
can be analyzed together downstream.

**What this notebook does:**
1. Loads the OPDC longitudinal dataset and attaches each participant's
   PPMI-style subgroup label (A/B/C).
2. Derives a `time_since_first_visit` field (years since each subject's
   first visit) to use in place of PPMI's visit-code `EVENT_ID`.
3. Looks up the PPMI MDS-UPDRS column names so the corresponding OPDC
   columns can be renamed to match them.
4. Selects and renames the OPDC Part I/II/III columns, and derives an
   ON/OFF medication-state flag from time since last levodopa dose.
5. Splits each part into ON- and OFF-state tables and pickles them to
   `data/02_processed/OPDC/`.

**Output files:**
- `P1ON_MDSUPDRS.pkl` / `P1OFF_MDSUPDRS.pkl` — Part I, ON/OFF state
- `P2ON_MDSUPDRS.pkl` / `P2OFF_MDSUPDRS.pkl` — Part II, ON/OFF state
- `P3ON_MDSUPDRS.pkl` / `P3OFF_MDSUPDRS.pkl` — Part III, ON/OFF state


In [39]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load OPDC Cohort and Subgroup Labels

In [40]:
df = pd.read_csv('../../data/01_raw/OPDC/OPDC_Discovery_PD_arm_longitudinal.csv', low_memory=False)

# OPDC subject IDs are stored as 'site/subjectnumber' (e.g. 'OX/1234');
# strip the site prefix so IDs are comparable to the subgroup file below.
df['subjid'] = [''.join(item.split('/')[1:]) for item in df['subjid'].astype(str)]

# Subgroup assignments use the same raw 0/1/2 coding as the PPMI subgroup
# file; relabel to the same A/B/C letters used there for consistency.
subgroups = pd.read_csv('./../../data/01_raw/OPDC/subgroups.csv')
subgroups['subgroup'] = subgroups['subgroup'].replace(0, 'A').replace(1, 'B').replace(2, 'C')

In [41]:
# Attach each participant's subgroup label. Note the OPDC subgroup file
# uses a differently-named ID column ('OPDC participant_id') than the
# longitudinal file ('subjid').
df = pd.merge(df, subgroups, left_on='subjid', right_on='OPDC participant_id')

### Derive Time-Since-First-Visit

OPDC doesn't use PPMI-style visit codes (`BL`, `V02`, ...), so we build a continuous time axis instead: the number of years elapsed since each subject's first recorded visit. This later stands in for PPMI's `EVENT_ID`.

In [42]:
df = df.sort_values(["subjid", "visit"]).reset_index(drop=True)

# Years elapsed since the previous visit for this subject (NaN on each
# subject's first visit, since there's no prior visit to diff against).
df["visit_interval_years"] = (df.groupby("subjid")["age"].diff())

# Cumulative years since the subject's first visit: treat the first
# visit's undefined interval as 0, then cumulatively sum the intervals.
df["time_since_first_visit"] = (
    df.groupby("subjid")["visit_interval_years"].transform(lambda x: x.fillna(0).cumsum())
)

### Load OFF-State Scores

**Note:** `df_off` is loaded and cleaned here but not merged into `df` or used anywhere later in this notebook — the ON/OFF split used further down is instead derived from time-since-last-levodopa (see below). Left in place in case it's needed for cross-checking; safe to remove if it's confirmed unused.

In [43]:
df_off = pd.read_csv('./../../data/01_raw/OPDC/OPDC_Discovery_PD_arm_extras_visit2_offscores.csv')

df_off['subjid'] = [''.join(item.split('/')[1:]) for item in df_off['subjid'].astype(str)]

# Strip the '_off' suffix so these column names match the ON-state column
# names used elsewhere (e.g. for potential comparison/merging).
df_off.columns = [col.replace('_off', '') for col in df_off.columns]

### Section 1 — Locate the MDS-UPDRS Columns

Quick check of which OPDC columns look UPDRS-related, to sanity-check the positional column selection below:

In [44]:
for col in df.columns:
    if 'updrs' in col.lower():
        print(col)

UPDRS_I
UPDRS_III
af_updrs_problem
UPDRS_IV
UPDRS_II


**Caution:** the ranges below select OPDC columns by *position*, not by name — they depend on the raw CSV's column order staying fixed. If the source file's columns are ever reordered, these slices will silently pick up the wrong fields.

In [45]:
p1_1_cols = list(df.columns[7:13])     # Part I, clinician-rated section
p1_2_cols = list(df.columns[204:211])  # Part I, patient-questionnaire section
p2_cols = list(df.columns[191:204])    # Part II, patient-questionnaire section
p3_cols = list(df.columns[14:47])      # Part III, motor exam

### Load PPMI Column Names for Renaming

To keep OPDC and PPMI data on a shared schema, the OPDC item columns selected above are renamed to their PPMI MDS-UPDRS equivalents. This cell just reads the PPMI source files far enough to grab their column names — the actual PPMI data isn't used here.

In [46]:
ppmi_mds_updrs_p1_1 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_I_10Aug2026.csv')
ppmi_mds_updrs_p1_1_cols = [col for col in ppmi_mds_updrs_p1_1.columns if 'NP1' in col]

ppmi_mds_updrs_p1_2 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_I_Patient_Questionnaire_10Aug2026.csv')
ppmi_mds_updrs_p1_2_cols = [col for col in ppmi_mds_updrs_p1_2.columns if 'NP1' in col]

ppmi_mds_updrs_p2 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS_UPDRS_Part_II__Patient_Questionnaire_10Aug2026.csv')
ppmi_mds_updrs_p2_cols = [col for col in ppmi_mds_updrs_p2.columns if 'NP2' in col]

ppmi_mds_updrs_p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')
ppmi_mds_updrs_p3_cols = [col for col in ppmi_mds_updrs_p3.columns if 'NP3' in col] + ['af_3_c1_last_lev', 'hoehn_yahr_stage']

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_90034/3334626334.py:10: DtypeWarning: Columns (16,21) have mixed types. Specify dtype option on import or set low_memory=False.
  ppmi_mds_updrs_p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')


### Build Part I, II, III Tables

For each part: select the relevant OPDC columns, rename them to the matching PPMI item codes, and derive a medication `State` flag from time since the last levodopa dose (`af_3_c1_last_lev`, given in minutes — converted to hours below). A subject is treated as `OFF` if 6+ hours have passed since their last dose, otherwise `ON`.

**Note:** the PPMI column-name lists are sliced with `[:-1]` (or `[:-3]` for Part III) when renaming — this drops trailing PPMI columns (e.g. total-score columns) that don't have an OPDC counterpart, so the remaining name lists line up positionally with the selected OPDC columns. This assumes the OPDC and (trimmed) PPMI column orders match item-for-item.

In [52]:
p1_final = df[['subjid', 'time_since_first_visit', 'af_3_c1_last_lev'] + [col for col in p1_1_cols + p1_2_cols]]
p1_final.columns = ['subjid', 'time_since_first_visit', 'af_3_c1_last_lev'] + ppmi_mds_updrs_p1_1_cols[:-1] + ppmi_mds_updrs_p1_2_cols[:-1]
p1_final = p1_final.rename(columns={'subjid': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'af_3_c1_last_lev': 'LastLevodopa'})
p1_final['LastLevodopa'] = p1_final['LastLevodopa'] / 60  # minutes -> hours
p1_final['State'] = ['OFF' if item >= 6 else 'ON' for item in p1_final['LastLevodopa']]
p1_final['EVENT_ID'] = p1_final['EVENT_ID']
p1_final[['PATNO', 'EVENT_ID', 'LastLevodopa']].to_csv('../../data/01_raw/OPDC/last_levodopa.csv')
p1_final.drop(columns='LastLevodopa', inplace=True)


p2_final = df[['subjid', 'time_since_first_visit', 'af_3_c1_last_lev'] + [col for col in p2_cols]]
p2_final.columns = ['subjid', 'time_since_first_visit', 'af_3_c1_last_lev'] + ppmi_mds_updrs_p2_cols[:-1]
p2_final = p2_final.rename(columns={'subjid': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'af_3_c1_last_lev': 'LastLevodopa'})
p2_final['LastLevodopa'] = p2_final['LastLevodopa'] / 60  # minutes -> hours
p2_final['State'] = ['OFF' if item >= 6 else 'ON' for item in p2_final['LastLevodopa']]
p2_final['EVENT_ID'] = p2_final['EVENT_ID']
p2_final.drop(columns='LastLevodopa', inplace=True)

p3_final = df[['subjid', 'time_since_first_visit', 'af_3_c1_last_lev', 'hoehn_yahr_stage'] + [col for col in p3_cols]]
p3_final.columns = ['subjid', 'time_since_first_visit', 'af_3_c1_last_lev', 'hoehn_yahr_stage'] + ppmi_mds_updrs_p3_cols[:-3]
p3_final = p3_final.rename(columns={'subjid': 'PATNO', 'time_since_first_visit': 'EVENT_ID', 'af_3_c1_last_lev': 'LastLevodopa', 'hoehn_yahr_stage': 'NHY'})
p3_final['LastLevodopa'] = p3_final['LastLevodopa'] / 60  # minutes -> hours
p3_final['State'] = ['OFF' if item >= 6 else 'ON' for item in p3_final['LastLevodopa']]
p3_final['EVENT_ID'] = p3_final['EVENT_ID']
p3_final.drop(columns='LastLevodopa', inplace=True)

### Split Each Part by Medication State and Save

For each part, split into ON- and OFF-state tables (dropping the now-redundant `State` column) and pickle both, mirroring the PPMI `P{n}ON_MDSUPDRS.pkl` / `P{n}OFF_MDSUPDRS.pkl` naming convention.

#### Part I

In [36]:
p1_final_tmp = p1_final.copy()

p1_final = p1_final_tmp[p1_final_tmp['State'] == 'ON'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P1ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p1_final, file)

p1_final = p1_final_tmp[p1_final_tmp['State'] == 'OFF'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P1OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p1_final, file)

#### Part II

In [37]:
p2_final_tmp = p2_final.copy()

p2_final = p2_final_tmp[p2_final_tmp['State'] == 'ON'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P2ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p2_final, file)

p2_final = p2_final_tmp[p2_final_tmp['State'] == 'OFF'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P2OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p2_final, file)

#### Part III

In [38]:
p3_final_tmp = p3_final.copy()

p3_final = p3_final_tmp[p3_final_tmp['State'] == 'ON'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P3ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3_final, file)

p3_final = p3_final_tmp[p3_final_tmp['State'] == 'OFF'].drop(columns=['State'])
with open('./../../data/02_processed/OPDC/P3OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3_final, file)